# Day 45: Naive Bayes Classifier

## The Idea
Naive Bayes is a probabilistic classifier built directly on Bayes' theorem: it estimates P(class | features) by combining how likely each feature value is within each class (P(features | class)) with how common each class is overall (P(class)), then picks whichever class has the highest resulting probability. The "naive" part is a simplifying assumption - it treats all features as conditionally independent given the class, meaning it multiplies together each feature's individual contribution rather than modeling how features interact. This assumption is rarely exactly true in real data, but it makes the math extremely fast and requires very little training data to estimate, and empirically it still works remarkably well for many tasks (especially text classification like spam detection). The cell below just loads the Iris dataset and splits it into train/test sets, same as previous days, so we have data to classify.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

X, y = load_iris(return_X_y=True)  # 4 continuous features, 3 flower species as classes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Training Gaussian Naive Bayes
There are several flavors of Naive Bayes depending on what kind of features you have (e.g. `MultinomialNB` for word counts in text). `GaussianNB` is the variant for continuous numeric features - it assumes that, within each class, each feature's values are normally (bell-curve) distributed, and it estimates the mean and variance of that distribution per feature per class from the training data. At prediction time, it plugs a new sample's feature values into these learned Gaussian distributions to get a likelihood for each class, combines that with Bayes' theorem, and outputs the most probable class. This is a very cheap model to train - essentially just computing means and variances - which is why Naive Bayes is often used as a fast baseline. The printed accuracy tells you the fraction of test flowers classified into the correct species.

In [ ]:
model = GaussianNB().fit(X_train, y_train)  # learns per-class mean/variance for each feature
print('Accuracy:', model.score(X_test, y_test))

## Predicted Probabilities
Because Naive Bayes is fundamentally computing P(class | features) via Bayes' theorem, it naturally produces a full probability distribution over all possible classes for each sample, not just a single predicted label - this is a genuine advantage over some other classifiers where probability estimates are an afterthought. `predict_proba` returns, for each sample, one probability per class that sums to 1. This is useful in practice when you care about model confidence - e.g. flagging predictions where the top probability is only slightly higher than the second, meaning the model is unsure. Here we look at the probabilities for just the first 5 test samples; the class with the highest probability in each row is what `.predict()` would return as the final label.

In [ ]:
probs = model.predict_proba(X_test[:5])  # probability of each of the 3 classes, per sample (rows sum to 1)
print(probs.round(3))

## Why 'Naive'?
The model is called "naive" because it assumes all features are conditionally independent given the class - i.e., knowing a flower's petal length tells you nothing extra about its petal width once you already know the species. In reality, features are very often correlated (in Iris, petal length and petal width are strongly related), so this assumption is technically violated. Yet Naive Bayes still frequently produces good classification accuracy in practice, because what matters for classification isn't a perfectly calibrated probability estimate - it's just getting the *ranking* of classes right, and the independence assumption often doesn't distort that ranking enough to change the predicted label. This is why Naive Bayes remains a popular, fast baseline for tasks like spam filtering even though its core assumption rarely holds exactly. The code below computes the correlation matrix between the 4 Iris features directly, so you can see for yourself how far from "independent" (i.e., near-zero correlation) they actually are.

In [ ]:
print('Feature correlations (violates the naive assumption if not near 0):')
import numpy as np
print(np.corrcoef(X.T).round(2))  # correlation matrix between the 4 features; large off-diagonal values show they aren't independent